# Course 4 - Applied Text Mining in Python
## Assignment 1: Date Extraction and Sorting with Regular Expressions

### Overview
This assignment applies regular expressions to extract and normalize dates from 500 messy medical notes. Each note contains exactly one date, but dates appear in many different formats (numeric, abbreviated, full month names, year-only, etc.). The extracted dates must be sorted in ascending chronological order, and the result is the sorted index sequence (not the dates themselves).

### Learning Objectives
- Write robust regular expressions to match diverse date formats
- Handle ambiguous and partial dates with normalization rules
- Parse and compare datetime objects
- Use Kendall's tau correlation as an evaluation metric for ranked outputs

### Dataset
- **File:** `assets/dates.txt` — 500 lines, each a medical note containing exactly one date
- **Output:** A pandas Series of length 500 containing the original row indices sorted chronologically

### Date Formats Encountered
| Format Example | Pattern Type |
|----------------|-------------|
| `04/20/2009`, `4/3/09` | Numeric MM/DD/YY or MM/DD/YYYY |
| `Mar-20-2009`, `Mar 20, 2009` | Month abbreviation + day + year |
| `20 Mar 2009`, `20 March, 2009` | Day-first with month name |
| `Mar 20th, 2009` | Ordinal day suffix (st/nd/rd/th) |
| `Feb 2009`, `Sep 2009` | Month + year only (day assumed = 1) |
| `6/2008`, `12/2009` | Month/year numeric (day assumed = 1) |
| `2009`, `2010` | Year only (Jan 1 assumed) |

### Normalization Rules
- `xx/xx/xx` format: assume MM/DD/YY
- Two-digit years: assume 1900s (e.g., `89` -> `1989`)
- Missing day: assume 1st of the month
- Missing month: assume January 1st of that year

### Assignment Structure
- **Single function:** `date_sorter()` — returns a Series of length 500 with dtype int, containing original row indices sorted by extracted date (tie-broken by original row number)

In [1]:
import pandas as pd

doc = []
with open('assets/dates.txt') as file:
    for line in file:
        doc.append(line)

df = pd.Series(doc)
df.head(10)

0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
5                    .Per 7/06/79 Movement D/O note:\n
6    4, 5/18/78 Patient's thoughts about current su...
7    10/24/89 CPT Code: 90801 - Psychiatric Diagnos...
8                         3/7/86 SOS-10 Total Score:\n
9             (4/10/71)Score-1Audit C Score Current:\n
dtype: object

In [2]:
import regex as re

# Month pattern component (used in compound patterns)
month = r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*'

# Pattern 1: Numeric dates MM/DD/YY(YY) or MM-DD-YY(YY)
pattern1 = r'(?P<month>\d{1,2})[/](?P<day>\d{1,2})[/](?P<year>\d{2,4})'

# Pattern 2: "Month DD, YYYY" or "Month DD YYYY" (e.g., Mar 20, 2009)
pattern2 = month + r'[\./-]?\s?(\d{1,2})[,\./-]?\s?[^;\[0-9](\d{2,4})'

# Pattern 3: "DD Month YYYY" (e.g., 20 Mar 2009)
pattern3 = r'(\d{1,2})[^(0-9|;)].?\s?/?-?' + month + r'?.?,?\s?/?-?(\d{2,4})'

# Pattern 4: Ordinal day (e.g., Mar 20th, 2009)
pattern4 = month + r'?.?\s?/?-?(\d{1,2})(?:st|nd|th).?\s?/?-?(\d{2,4})'

# Pattern 5: "Month YYYY" (e.g., Feb 2009)
pattern5 = r'[^0-9-/.]' + month + r'\s(\d{2,4})'

# Pattern 6: "M/YYYY" or "MM/YYYY"
pattern6 = r'[^/-;]*(\d{1,2})[/-](\d{4})[^;]*'

# Pattern 7: Year only (e.g., 2009)
pattern7 = r'[^/\.-a-z]?\s?(\d{4})[^/-0-9]?'

combined_pattern = "|".join([pattern1, pattern2, pattern3, pattern4, pattern5, pattern6, pattern7])
print("Pattern setup complete")

Pattern setup complete


In [3]:
pattern1_match = df.str.extractall(pat=pattern1)
# Show rows that matched pattern 1
matched_indices = [x[0] for x in pattern1_match.index.values]
print(f"Pattern 1 matched {len(set(matched_indices))} rows out of {len(df)}")
print(df[matched_indices[0]])

Pattern 1 matched 121 rows out of 500
03/25/93 Total time of visit (in minutes):



In [4]:
import re

text = ',wordp'

pattern = r'[,]word[p]'

pattern1= r'(?<=,)(word)(?=p)'

print(re.findall(pattern, text))
print(re.findall(pattern1, text))


[',wordp']
['word']


In [5]:
text7 = '2009; 2010b'

pattern = r'\d{4}[^;ab]*'
print(re.findall(pattern, text7))



['2009', '2010']


In [6]:
type(df)
df_proc = df.copy()
pattern1
print(df_proc.name) 
df_proc.name = 'text'
print(df_proc.name) 

None
text


In [7]:
import datetime 

#/= 'text'
df_proc = pd.DataFrame(df_proc)
print(pattern1)
print(df_proc.head(5))

df_proc['date'] = df_proc['text'].str.extract(pattern1)
print(df_proc.head(5))
mask = df_proc['date'].isna()

(?<=,)(word)(?=p)
                                                text
0       03/25/93 Total time of visit (in minutes):\n
1                     6/18/85 Primary Care Doctor:\n
2  sshe plans to move as of 7/8/71 In-Home Servic...
3              7 on 9/27/75 Audit C Score Current:\n
4  2/6/96 sleep studyPain Treatment Pain Level (N...
                                                text date
0       03/25/93 Total time of visit (in minutes):\n  NaN
1                     6/18/85 Primary Care Doctor:\n  NaN
2  sshe plans to move as of 7/8/71 In-Home Servic...  NaN
3              7 on 9/27/75 Audit C Score Current:\n  NaN
4  2/6/96 sleep studyPain Treatment Pain Level (N...  NaN


In [8]:
print(df_proc[mask]['text'].head(5))
a=df_proc[mask]['text'].str.extract(pattern3)
print(len(a))
a.head(5)
a.dropna( axis = 0, how = 'all', inplace = True)


0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
Name: text, dtype: object
500


## Function: date_sorter()

The `date_sorter()` function extracts a date from each of the 500 medical note lines and returns the row indices sorted in ascending chronological order. It applies regex patterns in priority order — most specific first, falling back to less specific patterns (month+year, year-only).

In [9]:
def date_sorter():
    import re
    import datetime as dt
    
    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    
    dates = []
    for i, text in enumerate(df):
        date = None
        
        # P1: Numeric MM/DD/YY or MM/DD/YYYY
        m = re.search(r'(\d{1,2})[/](\d{1,2})[/](\d{2,4})', text)
        if m:
            mo, day, yr = int(m.group(1)), int(m.group(2)), int(m.group(3))
            if yr < 100:
                yr += 1900
            try:
                date = dt.datetime(yr, mo, day)
            except ValueError:
                date = None
        
        if date is None:
            # P2: Month-name DD, YYYY  (e.g., Mar 20, 2009 or March 20 2009)
            m = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[\.,-]?\s*(\d{1,2})(?:st|nd|rd|th)?[,\.\s]++(\d{4})', text, re.IGNORECASE)
            if m:
                mo = month_map[m.group(1)[:3].capitalize()]
                day = int(m.group(2))
                yr = int(m.group(3))
                try:
                    date = dt.datetime(yr, mo, day)
                except ValueError:
                    date = None
        
        if date is None:
            # P3: DD Month-name YYYY  (e.g., 20 Mar 2009 or 20 March, 2009)
            m = re.search(r'(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[,.]?\s*(\d{4})', text, re.IGNORECASE)
            if m:
                day = int(m.group(1))
                mo = month_map[m.group(2)[:3].capitalize()]
                yr = int(m.group(3))
                try:
                    date = dt.datetime(yr, mo, day)
                except ValueError:
                    date = None
        
        if date is None:
            # P4: Month-name YYYY only  (e.g., Feb 2009)
            m = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(\d{4})', text, re.IGNORECASE)
            if m:
                mo = month_map[m.group(1)[:3].capitalize()]
                yr = int(m.group(2))
                date = dt.datetime(yr, mo, 1)
        
        if date is None:
            # P5: M/YYYY numeric  (e.g., 6/2008)
            m = re.search(r'(\d{1,2})[/](\d{4})', text)
            if m:
                mo = int(m.group(1))
                yr = int(m.group(2))
                if 1 <= mo <= 12:
                    date = dt.datetime(yr, mo, 1)
        
        if date is None:
            # P6: Year only  (e.g., 2009)
            m = re.search(r'(19\d{2}|20\d{2})', text)
            if m:
                yr = int(m.group(1))
                date = dt.datetime(yr, 1, 1)
        
        dates.append((i, date if date is not None else dt.datetime(1900, 1, 1)))
    
    # Sort by date, break ties by original row index
    dates.sort(key=lambda x: (x[1], x[0]))
    return pd.Series([x[0] for x in dates])

result = date_sorter()
assert len(result) == 500, f"Expected 500 entries, got {len(result)}"
assert result.dtype == int or result.dtype == 'int64', f"Must return int Series, got {result.dtype}"
print("date_sorter() returned Series of length", len(result))
print("First 5 indices:", result.head().tolist())


date_sorter() returned Series of length 500
First 5 indices: [25, 39, 40, 71, 233]
